# Benchmark Quantum Adder

Quantum arithmetic is a central ingredient in many quantum algorithms. Quantum circuits that manipulate encoded integers or numerical variables coherently rely on reversible arithmetic primitives such as addition, comparison, and multiplication. These building blocks appear prominently in Shor’s algorithm, where modular arithmetic is essential, and also in oracle-based settings related to Grover-style search, where arithmetic can be part of the problem encoding. In practice, arithmetic subroutines often contribute significantly to circuit depth and overall resource cost.

Among these primitives, the **quantum adder** is one of the simplest and most canonical examples. It implements the reversible transformation of one register by adding the value of another, making it a natural starting point for studying quantum arithmetic. It is also a basic component in more advanced arithmetic circuits, including multipliers, comparators, accumulators, and modular routines.

We benchmark an out-of-place addition by a constant:
$$
|x\rangle_n|0\rangle_n \rightarrow |x\rangle_n|x+c\, (\bmod 2^n) \rangle_n,
$$
where $c$ is a given integer.

#### The model
We start with a uniform superposition for $|x\rangle$, and add the constant $c=2^{\lfloor n/2 \rfloor}-1$:
$$
\frac{1}{\sqrt{2^n}}\sum_{x=0}^{2^n-1}|x\rangle_n|0\rangle_n \rightarrow \frac{1}{\sqrt{2^n}}\sum_{x=0}^{2^n-1}|x\rangle_n|x+c \, (\bmod 2^n)\rangle_n.
$$

#### The Scoring function
We measure the probabilty of getting a correct result:
$$
{\rm Score = }\sum_{y,x=0}^{2^n-1} P\left(y= x+c \, (\bmod 2^n)\right),
$$
where $y$ is the measured value of the second variable.

## Defining the model

In [2]:
import sys

sys.path.insert(0, "..")

from hardwares_preferences import HARDWARES

from benchmarking import *

from classiq import *

In [3]:
class Example4(BenchmarkExample):
    def __init__(self):
        super().__init__(
            name="Adder",
            num_qubits=4,
            constraints=Constraints(optimization_parameter="width"),
        )

    def create_main(self) -> callable:  # () -> main
        @qfunc
        def main(y: Output[QNum[self.num_qubits]], x: Output[QNum[self.num_qubits]]):
            allocate(y)
            allocate(x)
            hadamard_transform(x)
            y ^= x + 2 ** (y.size // 2) - 1

        return main

    async def submit(self, qprog: QuantumProgram) -> str:
        with ExecutionSession(qprog) as es:
            job = es.submit_sample()
            return job.id

    async def score(self, job_id):
        job = ExecutionJob.from_id(job_id)
        result = await job.result_async()
        df = result[0].value.dataframe

        mask = (
            (df["x"] + 2 ** (self.num_qubits // 2) - 1 - df["y"]) % 2**self.num_qubits
        ) == 0

        exec_minutes = (job.end_time - job.start_time).total_seconds() / 60.0

        return {
            "score": df.loc[mask, "probability"].sum(),
            "execution_time": exec_minutes,
        }

## Benchmarking a 4-qubits adder

Define a `BenchmarkExample` (the models are optimized for width):

In [4]:
example4 = Example4()

Define the number of shots and other hyperparameters:

In [5]:
common_config = {
    "max_timeout": 5e5,  # value is in seconds. Equals a bit more than 5 days."
    "num_shots": 1000,
}

Define `HardwareRunner` for each backend. Here we choose one machine for IBM, one for IonQ, as well as Classiq simulator for reference:

In [6]:
classiq_runners = [
    HardwareRunner("Classiq", backend_name, **common_config)
    for backend_name in HARDWARES["Classiq"][0:3]
]

# ionq_runners = [
#     HardwareRunner("IonQ", backend_name, **common_config)
#     for backend_name in HARDWARES["IonQ"][3:4]
# ]


# ibm_runners = [
#     HardwareRunner(
#         "IBM Quantum",
#         backend_name,
#         **common_config,
#         backend_kwargs={
#             "access_token": "PUT YOUR TOKEN HERE",
#             "channel": "PUT YOUR CHANNEL HERE",
#             "instance_crn": "PUT YOUR INSTANCE_CRN HERE",
#         },
#     )
#     for backend_name in HARDWARES["IBM Quantum"][0:1]
# ]

runners = [
    *classiq_runners,
    # *ionq_runners,
    # *ibm_runners,
]

In [7]:
print("Running for Backends:")
print(
    *[(runner.backend_service_provider, runner.backend_name) for runner in runners],
    sep="\n"
)

Running for Backends:
('Classiq', 'simulator')
('Classiq', 'simulator_statevector')
('Classiq', 'simulator_density_matrix')


Define a `ResultCollector` with a given benchmark filename:

In [8]:
FILENAME = "data/adder4.csv"

In [9]:
collector = ResultCollector(FILENAME, build_each_time=True)
await collector.reset_file()

In [10]:
header = "Here we go again"
print(
    "=" * 10
    + f"  {header} {example4.name}-{example4.num_qubits} ({datetime.datetime.now()})   "
    + "=" * 10
)
await asyncio.gather(*[collector.run(runner, example4) for runner in runners]);

==========  Here we go again Adder-4 (2026-03-13 00:08:33.310874)   ==========
2026-03-13 00:08:38.933863: Submit Adder-4 for Classiq - simulator
2026-03-13 00:08:39.175888: Submit Adder-4 for Classiq - simulator_statevector
2026-03-13 00:08:39.413722: Submit Adder-4 for Classiq - simulator_density_matrix
2026-03-13 00:08:40.237448: Completed Adder-4 for Classiq - simulator --> Score {'score': 1.0, 'execution_time': 0.010290983333333333}
2026-03-13 00:08:40.740156: Completed Adder-4 for Classiq - simulator_statevector --> Score {'score': 0.9999999999999998, 'execution_time': 0.012490433333333334}
** Report updated: Adder-4 for Classiq - simulator
** Report updated: Adder-4 for Classiq - simulator_statevector
2026-03-13 00:08:50.960810: Completed Adder-4 for Classiq - simulator_density_matrix --> Score {'score': 1.0, 'execution_time': 0.13879185}
** Report updated: Adder-4 for Classiq - simulator_density_matrix
